# Zero-shot VQA on RadImageNet-VQA: MedVLM-R1
Closed-ended (yes/no) and multiple-choice questions only. 500 questions, 5 groups x 100, seed 42.
Same prompts, sampling and scoring as the other three notebooks; only the model cells differ.

MedVLM-R1 was trained to reason in `<think>...</think><answer>X</answer>` format. If the smoke test shows outputs cut off before the answer (parsed as `unparsed`), raise `MAX_NEW_TOKENS` (e.g. 512) for this model only.

**Blank-image control:** all 500 sampled questions receive the same 512×512 RGB image. Change `BLANK_COLOR` to `"black"` to run the black control; results go to `blank_control_white/` or `blank_control_black/` rather than overwriting original CSVs.

## 1. Setup and Hugging Face login

In [6]:
!pip install -q -U transformers accelerate qwen-vl-utils datasets scikit-learn

from huggingface_hub import login
try:                                    # Colab
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except ImportError:                     # Kaggle
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))

In [7]:
import os, re, math, json
from PIL import Image
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_ID = "JZPeterPan/MedVLM-R1"
TAG = "medvlm_r1"                        # used in output file names
SEED = 42
MAX_NEW_TOKENS = 16                       # answer is one letter or yes/no
BLANK_COLOR = "white"                    # change to "black" for the black control
assert BLANK_COLOR in ("white", "black")
OUT_DIR = f"blank_control_{BLANK_COLOR}" # separate folders, same original filenames
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = os.path.join(OUT_DIR, f"vqa_results_{TAG}.csv")
SAMPLE_CSV = os.path.join(OUT_DIR, "vqa_sample.csv")             # same file for all four models
torch.manual_seed(SEED)

## 2. Load the data (closed + multiple_choice only)

In [8]:
ds = load_dataset("raidium/RadImageNet-VQA", "benchmark", split="test")

meta = ds.remove_columns("image").to_pandas()
meta["content_type"] = meta["metadata"].apply(lambda m: m["content_type"])
meta["anatomy"]      = meta["metadata"].apply(lambda m: m["location"])
meta["question_id"]  = meta["metadata"].apply(lambda m: m["question_id"])
meta["idx"] = np.arange(len(meta))        # row position in ds, used to fetch the image later

GROUPS = {   # (question_type, content_type) -> group name
    ("closed", "anatomy"):             "anatomy_yesno",
    ("closed", "pathology"):           "abnormality_yesno",   # "Does the X look abnormal?"
    ("closed", "pathology_specific"):  "pathology_yesno",
    ("multiple_choice", "anatomy"):    "anatomy_mcq",
    ("multiple_choice", "pathology"):  "pathology_mcq",
}
meta["group"] = [GROUPS.get((q, c)) for q, c in zip(meta.question_type, meta.content_type)]
pool = meta[meta.group.notna()].copy()    # drops the 2,000 open-ended questions
pool["answer"] = pool["answer"].str.strip()
pool.loc[pool.question_type == "closed", "answer"] = pool["answer"].str.lower()

print(len(pool), "questions in the pool")
print(pool.groupby("group").size())

README.md:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

benchmark/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  361MB            

benchmark/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/9000 [00:00<?, ? examples/s]

7000 questions in the pool
group
abnormality_yesno    1000
anatomy_mcq          1000
anatomy_yesno        2000
pathology_mcq        1000
pathology_yesno      2000
dtype: int64


## 3. Stratified sample (seed 42)
100 per group; each yes/no group is 50 `yes` + 50 `no`. The sample IDs are saved to a CSV (IDs only), and the same seed gives all four models identical questions.

In [9]:
closed = pool[pool.question_type == "closed"]
mcq    = pool[pool.question_type == "multiple_choice"]

parts = [g.groupby("answer").sample(50, random_state=SEED) for _, g in closed.groupby("group")]
parts += [g.sample(100, random_state=SEED) for _, g in mcq.groupby("group")]
sample = pd.concat(parts).sort_values("idx").reset_index(drop=True)

# smoke test: 1 question per group, never in the evaluation sample
smoke = pool.drop(index=pool.index[pool.idx.isin(sample.idx)]).groupby("group").sample(1, random_state=SEED)

# IDs only (no question text or answers): the dataset licence forbids redistribution
sample[["idx", "question_id", "group"]].to_csv(SAMPLE_CSV, index=False)

print(len(sample), "evaluation questions;", len(smoke), "smoke-test questions")
print(pd.crosstab(sample.group, sample.answer.where(sample.question_type == "closed", "MCQ")))

500 evaluation questions; 5 smoke-test questions
answer             MCQ  no  yes
group                          
abnormality_yesno    0  50   50
anatomy_mcq        100   0    0
anatomy_yesno        0  50   50
pathology_mcq      100   0    0
pathology_yesno      0  50   50


## 4. Load the model

In [10]:
import json
from huggingface_hub import hf_hub_download
from transformers import Qwen2VLForConditionalGeneration, Qwen2VLConfig, AutoProcessor
from qwen_vl_utils import process_vision_info

# MedVLM-R1's config.json has "use_cache": null; newer transformers requires a bool, so patch it before loading.
cfg_dict = json.load(open(hf_hub_download(MODEL_ID, "config.json")))
if cfg_dict.get("use_cache") is None:
    cfg_dict["use_cache"] = True
config = Qwen2VLConfig.from_dict(cfg_dict)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, config=config, torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa")
try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
except Exception as e:                      # fall back to the base model's processor if the repo has none
    print("processor not in MedVLM-R1 repo, using Qwen2-VL-2B-Instruct's:", type(e).__name__)
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
model.eval()
print(model.device)

config.json:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

... (6 outputs left out)

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

cuda:0


## 5. Prompts, inference and parsing

In [11]:
LETTERS = "ABCD"

def build_prompt(row):
    if row["question_type"] == "multiple_choice":
        opts = "\n".join(f"{L}. {c}" for L, c in zip(LETTERS, row["choices"]))
        return ("You are a medical assistant. Look at this image and answer the following multiple-choice "
                "question with ONLY the letter of the correct option (e.g. 'B'), nothing else.\n\n"
                f"Question: {row['question']}\nOptions:\n{opts}")
    return ("You are a medical assistant. Look at this image and answer the following question "
            "with ONLY the single word 'Yes' or 'No'.\n\n"
            f"Question: {row['question']}")

def predict(image, prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

def parse_mcq(output):
    m = re.search(r"\b([A-D])\b", output)          # first standalone option letter
    return m.group(1) if m else "unparsed"

def parse_yesno(output):
    m = re.search(r"\b(yes|no)\b", output.lower())  # whichever appears first
    return m.group(1) if m else "unparsed"

def parse(row, output):
    m = re.search(r"<answer>(.*?)</answer>", output, re.S | re.I)   # MedVLM-R1 may answer as <answer>X</answer>
    text = m.group(1) if m else output
    return parse_mcq(text) if row["question_type"] == "multiple_choice" else parse_yesno(text)

def run_one(row):
    image = Image.new("RGB", (512, 512), (255, 255, 255) if BLANK_COLOR == "white" else (0, 0, 0))
    raw = predict(image, build_prompt(row))
    return raw, parse(row, raw)

## 6. Smoke test (5 questions, not in the results)
Check that outputs parse, nothing runs out of memory, and the time per question.

In [12]:
import time
for _, r in smoke.iterrows():
    t = time.time()
    raw, pred = run_one(r)
    print(f"[{r.group}] true={r.answer!r} pred={pred!r} raw={raw!r} ({time.time()-t:.1f}s)")

[abnormality_yesno] true='yes' pred='no' raw='No' (3.9s)
[anatomy_mcq] true='C' pred='B' raw='B' (1.3s)
[anatomy_yesno] true='yes' pred='yes' raw='Yes' (1.3s)
[pathology_mcq] true='C' pred='C' raw='C' (1.3s)
[pathology_yesno] true='no' pred='no' raw='No' (1.3s)


## 7. Run the 500 questions (resumes if the session restarts)

In [13]:
done = pd.read_csv(OUT_CSV) if os.path.exists(OUT_CSV) else pd.DataFrame()
rows = done.to_dict("records")
seen = set(done["idx"]) if len(done) else set()

for _, r in tqdm(sample.iterrows(), total=len(sample)):
    if r["idx"] in seen:
        continue
    raw, pred = run_one(r)
    rows.append({"idx": r["idx"], "question_id": r["question_id"], "group": r["group"],
                 "anatomy": r["anatomy"], "question": r["question"],
                 "true_answer": r["answer"], "predicted_answer": pred, "raw_output": raw})
    if len(rows) % 50 == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False)

results = pd.DataFrame(rows)
results.to_csv(OUT_CSV, index=False)
print(len(results), "rows saved to", OUT_CSV)

  0%|          | 0/500 [00:00<?, ?it/s]

500 rows saved to blank_control_white/vqa_results_medvlm_r1.csv


## 8. Score
Unparsed outputs count as wrong and stay in the denominator.

In [14]:
def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h

results["correct"] = results.true_answer == results.predicted_answer
n, k = len(results), int(results.correct.sum())
lo, hi = wilson(k, n)
print(f"Overall accuracy: {k}/{n} = {k/n:.3f}  (95% CI {lo:.3f}-{hi:.3f})")
print(f"Parsed cleanly:   {(results.predicted_answer != 'unparsed').sum()}/{n}\n")

by_group = results.groupby("group").correct.agg(["sum", "count", "mean"]).rename(columns={"sum": "correct", "count": "n", "mean": "accuracy"})
print(by_group.round(3), "\n")
print(results.groupby("anatomy").correct.agg(["count", "mean"]).round(3).rename(columns={"count": "n", "mean": "accuracy"}), "\n")

yn = results[results.group.str.endswith("yesno")]
print(classification_report(yn.true_answer, yn.predicted_answer, labels=["yes", "no"], zero_division=0))
print(pd.DataFrame(confusion_matrix(yn.true_answer, yn.predicted_answer, labels=["yes", "no", "unparsed"]),
                   index=["true yes", "true no", "true unparsed"], columns=["pred yes", "pred no", "pred unparsed"]))

by_group.to_csv(os.path.join(OUT_DIR, f"vqa_scores_{TAG}.csv"))

Overall accuracy: 219/500 = 0.438  (95% CI 0.395-0.482)
Parsed cleanly:   500/500

                   correct    n  accuracy
group                                    
abnormality_yesno       50  100      0.50
anatomy_mcq             36  100      0.36
anatomy_yesno           57  100      0.57
pathology_mcq           25  100      0.25
pathology_yesno         51  100      0.51 

              n  accuracy
anatomy                  
abdomen     163     0.589
ankle foot   52     0.327
brain        31     0.387
hip          38     0.395
knee         48     0.271
lung        116     0.388
shoulder     26     0.423
spine        26     0.385 

              precision    recall  f1-score   support

         yes       0.61      0.15      0.24       150
          no       0.52      0.90      0.66       150

    accuracy                           0.53       300
   macro avg       0.56      0.53      0.45       300
weighted avg       0.56      0.53      0.45       300

               pred yes  pred no

## 9. Misclassified cases

In [15]:
wrong = results[~results.correct][["group", "question", "true_answer", "predicted_answer", "raw_output"]]
print(len(wrong), "wrong")
wrong.head(30)

281 wrong


,group,question,true_answer,predicted_answer,raw_output
0,abnormality_yesno,Does this scan reveal any abnormalities?,yes,no,No
1,anatomy_yesno,Can you see the ankle foot?,yes,no,No
2,anatomy_mcq,Select the correct anatomical region.,D,C,C
3,pathology_mcq,What is the most likely pathology in this image?,C,B,B
5,anatomy_mcq,Select the correct anatomical region.,B,D,D
6,pathology_mcq,Select the pathology that best describes this ...,B,A,A
7,pathology_yesno,Does the patient have pancreatic lesion?,yes,no,No
8,anatomy_yesno,Is the knee visible in the image?,yes,no,No
9,anatomy_mcq,Which of the following anatomical regions is d...,D,C,C
10,pathology_mcq,What pathology is shown in this scan of abdomen,C,D,D


In [17]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

# Locate output folders
root = Path.cwd()
folders = [
    root / "blank_control_white",
    root / "blank_control_black",
]

# Include only existing folders
folders = [folder for folder in folders if folder.is_dir()]

if not folders:
    raise FileNotFoundError("No blank-control output folders found!")

# Create ZIP
zip_path = root / "medvlmr1_blank_control_results.zip"

with ZipFile(zip_path, "w", ZIP_DEFLATED) as zf:
    for folder in folders:
        for file in folder.rglob("*"):
            if file.is_file():
                zf.write(file, arcname=file.relative_to(root))

print(f"ZIP created: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024:.2f} KB")

# Download to your computer
files.download(str(zip_path))

ZIP created: /content/medvlmr1_blank_control_results.zip
Size: 8.18 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>